# Face Recognition dengan OpenCV dan Backpropagation

Notebook ini membangun contoh proyek face recognition memakai dataset AT&T face. Pipeline yang dipakai adalah:

1. Download dan load dataset AT&T.
2. Preprocessing gambar dengan OpenCV.
3. Training neural network sederhana dengan backpropagation.
4. Evaluasi model.
5. Simpan artifacts untuk deployment.

## Tujuan

Notebook ini dibuat agar bisa dipakai langsung untuk tugas kuliah dan juga menjadi dasar deployment menggunakan Streamlit atau Docker.

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'requirements.txt').exists():
    candidate_root = PROJECT_ROOT.parent
    if (candidate_root / 'requirements.txt').exists():
        PROJECT_ROOT = candidate_root

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')])

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

sys.path.insert(0, str(PROJECT_ROOT))

from src.att_faces import download_att_faces, load_att_faces
from src.pipeline import load_artifacts, predict_bgr_image, save_artifacts, train_face_recognition

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

In [ ]:
data_dir = PROJECT_ROOT / 'data' / 'att_faces'
download_att_faces(data_dir)
images, labels = load_att_faces(data_dir, image_size=(64, 64))

print(f'Total images: {len(images)}')
print(f'Unique subjects: {len(np.unique(labels))}')

fig, axes = plt.subplots(3, 4, figsize=(10, 7))
rng = np.random.default_rng(42)
indices = rng.choice(len(images), size=12, replace=False)

for ax, index in zip(axes.ravel(), indices):
    ax.imshow(images[index], cmap='gray')
    ax.set_title(labels[index])
    ax.axis('off')

plt.tight_layout()

In [ ]:
artifacts, metrics, history, split = train_face_recognition(
    data_dir=data_dir,
    image_size=(64, 64),
    hidden_size=128,
    learning_rate=0.01,
    epochs=80,
    batch_size=32,
    test_size=0.25,
    random_state=42,
)

metrics

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
ax_loss.plot(history.losses, color='steelblue')
ax_loss.set_title('Training Loss')
ax_loss.set_xlabel('Epoch')
ax_loss.set_ylabel('Loss')

ax_acc.plot(history.accuracies, color='darkorange')
ax_acc.set_title('Training Accuracy')
ax_acc.set_xlabel('Epoch')
ax_acc.set_ylabel('Accuracy')

plt.tight_layout()

In [ ]:
X_train, X_test, y_train, y_test = split
y_pred = artifacts.model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print(f'Test accuracy: {test_accuracy:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=artifacts.label_encoder.classes_, digits=4))

In [ ]:
output_dir = PROJECT_ROOT / 'artifacts'
save_artifacts(artifacts, output_dir)
reloaded_artifacts = load_artifacts(output_dir)

sample_index = 0
sample_image = images[sample_index]
sample_bgr = cv2.cvtColor(sample_image, cv2.COLOR_GRAY2BGR)
predicted_label, confidence, probabilities = predict_bgr_image(sample_bgr, reloaded_artifacts)

print(f'Sample label: {labels[sample_index]}')
print(f'Prediction: {predicted_label}')
print(f'Confidence: {confidence:.2%}')

top_indices = np.argsort(probabilities)[::-1][:5]
for index in top_indices:
    subject = reloaded_artifacts.label_encoder.inverse_transform([index])[0]
    print(f'{subject}: {probabilities[index]:.4f}')

## Deployment

Setelah artifact tersimpan di folder `artifacts/`, jalankan deployment lokal dengan:

```bash
streamlit run app.py
```

Atau jalankan container Docker:

```bash
docker build -t att-face-recognition .
docker run -p 8501:8501 att-face-recognition
```